In [1]:
# eda news

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

import nltk
from nltk.corpus import stopwords
nltk.download("stopwords")

sns.set(style="whitegrid")

df = pd.read_csv("../data/raw_analyst_ratings.csv")

df.head()


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\samha\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,Unnamed: 0,headline,url,publisher,date,stock
0,0,Stocks That Hit 52-Week Highs On Friday,https://www.benzinga.com/news/20/06/16190091/s...,Benzinga Insights,2020-06-05 10:30:54-04:00,A
1,1,Stocks That Hit 52-Week Highs On Wednesday,https://www.benzinga.com/news/20/06/16170189/s...,Benzinga Insights,2020-06-03 10:45:20-04:00,A
2,2,71 Biggest Movers From Friday,https://www.benzinga.com/news/20/05/16103463/7...,Lisa Levin,2020-05-26 04:30:07-04:00,A
3,3,46 Stocks Moving In Friday's Mid-Day Session,https://www.benzinga.com/news/20/05/16095921/4...,Lisa Levin,2020-05-22 12:45:06-04:00,A
4,4,B of A Securities Maintains Neutral on Agilent...,https://www.benzinga.com/news/20/05/16095304/b...,Vick Meyer,2020-05-22 11:38:59-04:00,A


In [ ]:
# Convert date to datetime with UTC offset handled
df["date"] = pd.to_datetime(df["date"], utc=True, errors="coerce")

# Drop rows with missing headlines
df = df.dropna(subset=["headline"])

# Remove odd spaces
df["headline"] = df["headline"].str.strip()


In [ ]:
df["headline_length"] = df["headline"].apply(len)

df["headline_length"].describe()


In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(df["headline_length"], bins=40, kde=True)
plt.title("Distribution of Headline Lengths")
plt.xlabel("Length of headline")
plt.ylabel("Count")
plt.show()


In [ ]:
publisher_counts = df["publisher"].value_counts()

publisher_counts.head(20)


In [ ]:
plt.figure(figsize=(12,6))
publisher_counts.head(15).plot(kind="bar")
plt.title("Top 15 Most Active Publishers")
plt.ylabel("Article Count")
plt.xticks(rotation=45)
plt.show()


In [ ]:
df_daily = df.groupby(df["date"].dt.date).size()

plt.figure(figsize=(14,5))
df_daily.plot()
plt.title("Daily News Publication Trend")
plt.ylabel("Number of Articles")
plt.show()


In [ ]:
df["day_of_week"] = df["date"].dt.day_name()

sns.countplot(data=df, x="day_of_week", order=[
    "Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"
])
plt.title("Articles by Day of Week")
plt.xticks(rotation=45)
plt.show()


In [ ]:
df["hour"] = df["date"].dt.hour

plt.figure(figsize=(12,5))
sns.countplot(data=df, x="hour")
plt.title("Articles by Hour of Publication (UTC-4)")
plt.show()


In [ ]:
stop = stopwords.words("english")
vectorizer = CountVectorizer(
    stop_words=stop,
    max_features=40
)

bag = vectorizer.fit_transform(df["headline"])
word_freq = dict(zip(vectorizer.get_feature_names_out(), bag.sum(axis=0).A1))

word_freq


In [ ]:
plt.figure(figsize=(12,6))
pd.Series(word_freq).sort_values(ascending=False).plot(kind="bar")
plt.title("Top Keywords in Headlines")
plt.xticks(rotation=75)
plt.show()


In [ ]:
vectorizer = CountVectorizer(
    max_df=0.9,
    min_df=10,
    stop_words="english"
)
X = vectorizer.fit_transform(df["headline"])

lda = LatentDirichletAllocation(n_components=5, random_state=42)
lda.fit(X)

for idx, topic in enumerate(lda.components_):
    print(f"\nTOPIC #{idx+1}:")
    print([vectorizer.get_feature_names_out()[i] for i in topic.argsort()[-10:]])


In [ ]:
def extract_domain(x):
    if isinstance(x, str) and "@" in x:
        return x.split("@")[-1]
    return "unknown"

df["publisher_domain"] = df["publisher"].apply(extract_domain)

df["publisher_domain"].value_counts().head(10)


In [ ]:
df.to_csv("../data/news_cleaned.csv", index=False)
